# IEEE 118-Bus System - Pre-Processing

This notebook loads the IEEE 118-bus system directly from the **MATPOWER case118** CSV files and prepares the data needed for the branch-oriented **SC-DCOPF** formulation.

## 1. Imports

In [5]:
import numpy as np
import pandas as pd
import networkx as nx

## 2. Load the MATPOWER case118 CSV files

Raw MATPOWER data (1-indexed buses). Column meanings:
- **bus**: `bus_i`, `type` (3 = reference/slack), `Pd` (active load, MW), ...
- **gen**: `bus`, `Pmax`, `Pmin` (MW), `status`, ...
- **gencost**: `c2`, `c1`, `c0` (polynomial cost); aligns **row-by-row** with `gen`.
- **branch**: `fbus`, `tbus`, `x` (p.u.), `rateA` (MVA), `status`, ...

In [6]:
DATA_DIR = "data/matpower"   # raw MATPOWER case118 CSVs
BASE_MVA = 100.0   # MATPOWER case118 system base

bus_df     = pd.read_csv(f"{DATA_DIR}/bus_data_118-bus.csv")
gen_df     = pd.read_csv(f"{DATA_DIR}/gen_data_118-bus.csv")
gencost_df = pd.read_csv(f"{DATA_DIR}/gencost_data_118-bus.csv")
branch_df  = pd.read_csv(f"{DATA_DIR}/branch_data_118-bus.csv")

print(f"Buses      : {len(bus_df)}")
print(f"Generators : {len(gen_df)}")
print(f"Gen costs  : {len(gencost_df)}")
print(f"Branches   : {len(branch_df)}")

Buses      : 118
Generators : 54
Gen costs  : 54
Branches   : 186


### 2.1 Sanity checks

Before building anything we confirm the assumptions the whole model relies on:
1. `gencost` aligns 1-to-1 with `gen` (same number of rows).
2. Exactly one reference/slack bus (`type == 3`), and it is bus 69.
3. The slack bus is a real generator (`Pmax > 0`).

In [7]:
# 1. gencost aligns row-by-row with gen
assert len(gencost_df) == len(gen_df), (
    f"gencost rows ({len(gencost_df)}) != gen rows ({len(gen_df)}); "
    "cost/generator alignment would be wrong."
)

# 2. Exactly one reference bus (type == 3)
ref_buses = bus_df.loc[bus_df["type"] == 3, "bus_i"].tolist()
assert len(ref_buses) == 1, f"Expected exactly one slack bus, found {ref_buses}"
slack_bus = int(ref_buses[0])

# 3. The slack is a real generator
slack_gen = gen_df[gen_df["bus"] == slack_bus]
assert not slack_gen.empty, f"Slack bus {slack_bus} is not in the generator list!"

print(f"Slack bus (reference) : {slack_bus}")
print(f"Slack Pmax            : {slack_gen['Pmax'].iloc[0]:.0f} MW")
print(f"System base           : {BASE_MVA:.0f} MVA")
print("\nAll sanity checks passed.")

Slack bus (reference) : 69
Slack Pmax            : 1182 MW
System base           : 100 MVA

All sanity checks passed.


## 3. Convert to per-unit and build branch-oriented structures

Three DataFrames map 1-to-1 onto the SC-DCOPF notation:
- `gens`   — one row per **real** generator ($g \in \mathcal{G}$), with `pmin_pu`, `pmax_pu`, `c1`.
- `loads`  — one row per bus ($i \in \mathcal{N}$), with `pd_pu` ($P_{d,i}$).
- `branches` — one row per **branch** ($(i,j) \in \mathcal{L}$), with `x_pu`, `rate_pu`.

**Per-unit rules:** powers and ratings are divided by `BASE_MVA`;
reactance `x` is *already* p.u. in MATPOWER and is kept as-is. Cost `c1` stays in \$/MWh
(the base-MVA factor is handled later, in the model objective).

### 3.1 Generators

Only in-service generators with `Pmax > 0` are kept. The 35 synchronous condensers
(`Pmax == 0`) do nothing in a DC model (they only provide reactive power), so they are
dropped. Cost `c1` is taken
by **row position** before filtering.

In [8]:
gen_raw = gen_df.reset_index(drop=True).copy()
gen_raw["c1"] = gencost_df.reset_index(drop=True)["c1"]   # positional alignment
gen_raw = gen_raw[gen_raw["status"] == 1]                 # in-service only

real_mask = gen_raw["Pmax"] > 0
n_dropped = int((~real_mask).sum())

gens = pd.DataFrame({
    "gen_id":  range(int(real_mask.sum())),
    "bus":     gen_raw.loc[real_mask, "bus"].astype(int).values,
    "pmin_pu": gen_raw.loc[real_mask, "Pmin"].values / BASE_MVA,
    "pmax_pu": gen_raw.loc[real_mask, "Pmax"].values / BASE_MVA,
    "c1":      gen_raw.loc[real_mask, "c1"].values,        # $/MWh
})

assert slack_bus in set(gens["bus"]), "Slack generator was dropped — check filter!"

print(f"Real generators kept (Pmax>0)          : {len(gens)}")
print(f"Synchronous condensers dropped (Pmax==0): {n_dropped}")
print(f"Total installed capacity                : {gens['pmax_pu'].sum() * BASE_MVA:.0f} MW")
gens.head()

Real generators kept (Pmax>0)          : 19
Synchronous condensers dropped (Pmax==0): 35
Total installed capacity                : 6515 MW


,gen_id,bus,pmin_pu,pmax_pu,c1
0,0,10,0.0,5.05,24.983420
1,1,12,0.0,0.85,124.581564
2,2,25,0.0,2.21,28.948321
3,3,26,0.0,4.85,22.220980
4,4,31,0.0,0.17,25.993982


### 3.2 Loads ($P_{d,i}$) — one entry per bus

Every bus gets a load value (0 if it has no demand), so the energy balance can be written
for all $i \in \mathcal{N}$.

In [9]:
loads = pd.DataFrame({
    "bus":   bus_df["bus_i"].astype(int).values,
    "pd_pu": bus_df["Pd"].values / BASE_MVA,
})

print(f"Buses          : {len(loads)}")
print(f"Buses with load: {(loads['pd_pu'] > 0).sum()}")
print(f"Total demand   : {loads['pd_pu'].sum() * BASE_MVA:.1f} MW")
loads.head()

Buses          : 118
Buses with load: 99
Total demand   : 4242.0 MW


,bus,pd_pu
0,1,0.51
1,2,0.20
2,3,0.39
3,4,0.39
4,5,0.00


### 3.3 Branches ($\mathcal{L}$)

One row per **branch** (lines *and* transformers). Parallel circuits between the same pair of buses are kept as **separate rows** with
distinct `branch_id`, because each carries its own flow variable $f_{ij,c}$. The `label`
(`from-to`) is *not* a unique key when parallels exist — `branch_id` is.

In [10]:
br = branch_df[branch_df["status"] == 1].reset_index(drop=True).copy()

branches = pd.DataFrame({
    "branch_id": range(len(br)),
    "from_bus":  br["fbus"].astype(int).values,
    "to_bus":    br["tbus"].astype(int).values,
    "x_pu":      br["x"].values,               # already p.u. — do NOT convert
    "rate_pu":   br["rateA"].values / BASE_MVA,
})
branches["label"] = (branches["from_bus"].astype(str) + "-"
                     + branches["to_bus"].astype(str))

# integrity checks
assert (branches["x_pu"] > 0).all(), "Zero/negative reactance found."
assert branches["from_bus"].between(1, 118).all()
assert branches["to_bus"].between(1, 118).all()

n_parallel = int(branches.duplicated(subset=["from_bus", "to_bus"], keep=False).sum())
print(f"Branches (in service): {len(branches)}")
print(f"Branches in a parallel group: {n_parallel}")
branches.head()

Branches (in service): 186
Branches in a parallel group: 14


,branch_id,from_bus,to_bus,x_pu,rate_pu,label
0,0,1,2,0.09990,1.51,1-2
1,1,1,3,0.04240,1.51,1-3
2,2,4,5,0.00798,1.76,4-5
3,3,3,5,0.10800,1.75,3-5
4,4,5,6,0.05400,1.76,5-6


## 4. Identify radial lines for the contingency set $\mathcal{C}_T$

The transmission contingency set $\mathcal{C}_T$ contains only **non-radial** line outages.
A *bridge* is a line whose removal disconnects the network, isolating one or more buses —
such an outage would leave the isolated bus with a free angle and an infeasible balance, so
it must be excluded from $\mathcal{C}_T$.

We detect bridges with `networkx` on a **`MultiGraph`** (not a simple `Graph`). This matters
because of **parallel circuits**: two lines between the same pair of buses are individually
safe to remove (the parallel keeps the buses connected), so neither is a bridge. A simple
graph would collapse the parallels into one edge and wrongly flag it. The `MultiGraph`
keeps every circuit as a distinct edge, keyed by `branch_id`, so `nx.bridges` is correct.

In [11]:
# Buses = nodes, branches = edges. MultiGraph keeps parallel circuits separate.
G = nx.MultiGraph()
G.add_nodes_from(bus_df["bus_i"].astype(int).tolist())
for row in branches.itertuples(index=False):
    G.add_edge(row.from_bus, row.to_bus, key=row.branch_id) # The edge key is branch_id, to map a bridge edge back to its row.

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Fully connected: {nx.is_connected(G)}")

# Bridges come back as (u, v) endpoint-bus pairs of each radial branch.
bridge_pairs = set(nx.bridges(G))

# A branch is radial if its two buses match a bridge pair (test both directions).
def _is_bridge(u, v):
    return (u, v) in bridge_pairs or (v, u) in bridge_pairs

branches["is_bridge"] = [
    _is_bridge(r.from_bus, r.to_bus) for r in branches.itertuples(index=False)
]

# A parallel circuit must never be a bridge.
par_mask = branches.duplicated(subset=["from_bus", "to_bus"], keep=False)
assert not branches.loc[par_mask, "is_bridge"].any(), "A parallel line was flagged as a bridge!"

n_bridge = int(branches["is_bridge"].sum())
print(f"\nBridges (radial lines, excluded from C_T): {n_bridge}")
print(f"Non-radial lines (eligible for C_T)      : {len(branches) - n_bridge}")
branches[branches["is_bridge"]][["branch_id", "from_bus", "to_bus", "label"]]

Graph: 118 nodes, 186 edges
Fully connected: True

Bridges (radial lines, excluded from C_T): 9
Non-radial lines (eligible for C_T)      : 177


,branch_id,from_bus,to_bus,label
6,6,8,9,8-9
8,8,9,10,9-10
112,112,71,73,71-73
132,132,85,86,85-86
133,133,86,87,86-87
175,175,110,111,110-111
176,176,110,112,110-112
182,182,68,116,68-116
183,183,12,117,12-117


## 5. Export processed data

The clean tables are written to a separate **`processed/`** folder, keeping the raw MATPOWER
inputs (`data/`) untouched. The SC-DCOPF model notebook reads only from `processed/`.

| File | Contents |
|---|---|
| `gens.csv`        | real generators ($\mathcal{G}$): `bus`, `pmin_pu`, `pmax_pu`, `c1` |
| `loads.csv`       | per-bus demand ($P_{d,i}$): `bus`, `pd_pu` |
| `branches.csv`    | branches ($\mathcal{L}$): endpoints, `x_pu`, `rate_pu`, `is_bridge` |
| `system_meta.csv` | `base_mva`, `slack_bus` |

In [12]:
import os

OUT_DIR = "processed/matpower"
os.makedirs(OUT_DIR, exist_ok=True)

gens.to_csv(f"{OUT_DIR}/gens.csv", index=False)
loads.to_csv(f"{OUT_DIR}/loads.csv", index=False)
branches.to_csv(f"{OUT_DIR}/branches.csv", index=False)

meta = pd.DataFrame({
    "key":   ["base_mva", "slack_bus"],
    "value": [BASE_MVA,   slack_bus],
})
meta.to_csv(f"{OUT_DIR}/system_meta.csv", index=False)

print(f"Wrote 4 files to {OUT_DIR}/:")
print(f"  gens.csv        ({len(gens)} generators)")
print(f"  loads.csv       ({len(loads)} buses)")
print(f"  branches.csv    ({len(branches)} branches)")
print(f"  system_meta.csv (base_mva={BASE_MVA:.0f}, slack_bus={slack_bus})")

Wrote 4 files to processed/:
  gens.csv        (19 generators)
  loads.csv       (118 buses)
  branches.csv    (186 branches)
  system_meta.csv (base_mva=100, slack_bus=69)
